In [2]:
from src.dataset import load_data
from src.model import load_model
from src.inference import predict
from src.evaluation import compute_metrics, analyze_errors
from src.benchmark import benchmark_model
from src.reporting import (
    compare_models,
    compare_performance,
    export_results,
    plot_model_metrics,
    plot_throughput
)

In [ ]:
MODELS = {
    "DistilBERT": (
        "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
    ),
    "BERT": (
        "textattack/bert-base-uncased-SST-2"
    )
}

DATASET_NAME = "stanfordnlp/sst2"




In [ ]:

RUN_FULL_BENCHMARK = True

MAX_EXAMPLES = 872 if RUN_FULL_BENCHMARK else 20

INFERENCE_BATCH_SIZE = 8

BENCHMARK_BATCH_SIZES = (
    [1, 4, 8, 16] if RUN_FULL_BENCHMARK else [8]
)

BENCHMARK_REPETITIONS = (
    3 if RUN_FULL_BENCHMARK else 1
)


In [ ]:


dataset = load_data(DATASET_NAME, MAX_EXAMPLES)


metrics_by_model = {}
benchmark_by_model = {}
errors_by_model = {}


for model_name, model_id in MODELS.items():

        print(f"Evaluating {model_name}...")


        tokenizer ,model ,id2label =load_model(model_id)

        labels,confidences=predict(dataset , tokenizer,model,INFERENCE_BATCH_SIZE)

        print(f"Evaluating: {model_name}")
        print("Label mapping:", id2label)

        if model_name == "DistilBERT":
            assert id2label[0] == "NEGATIVE"
            assert id2label[1] == "POSITIVE"

        elif model_name == "BERT":
            print("BERT uses SST-2 classification.")
            print("Checking predictions against reference sentences...")

            test_sentences = [
                    "This movie was absolutely wonderful!",
                    "This was a terrible and disappointing film."
                ]


            from datasets import Dataset

            test_dataset = Dataset.from_dict({
                "sentence": test_sentences
            })

            test_labels, _ = predict(
                test_dataset, tokenizer, model, batch_size=2
            )

            print("Sanity-check predictions:", test_labels)

        metrics=compute_metrics(dataset['label'] , labels)

        errors=analyze_errors(dataset,labels,confidences)

        benchmark=benchmark_model(dataset,tokenizer,model,batch_sizes=BENCHMARK_BATCH_SIZES,repetitions=BENCHMARK_REPETITIONS)

        metrics_by_model[model_name] = metrics
        errors_by_model[model_name] = errors
        benchmark_by_model[model_name] = benchmark


model_comparison=compare_models(metrics_by_model)

performance_comparison=compare_performance(benchmark_by_model)

export_results(model_comparison,performance_comparison,errors_by_model["DistilBERT"]["errors"],errors_by_model["BERT"]["errors"])

# TODO: plot_model_metrics()
plot_model_metrics(model_comparison)

plot_throughput(performance_comparison)


Evaluating DistilBERT...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Evaluating: DistilBERT
Label mapping: {0: 'NEGATIVE', 1: 'POSITIVE'}
Sanity-check predictions: [1, 0]
Evaluating BERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Evaluating: BERT
Label mapping: {0: 'LABEL_0', 1: 'LABEL_1'}
BERT uses SST-2 classification.
Checking predictions against reference sentences...
Sanity-check predictions: [1, 0]


In [ ]:
#For 20 examples
assert len(metrics_by_model) == 2
assert len(benchmark_by_model) == 2
assert len(errors_by_model) == 2

assert len(model_comparison) == 2
assert len(performance_comparison) == 2
from pathlib import Path

required_files = [
    "results/model_comparison.csv",
    "results/performance_comparison.csv",
    "results/distilbert_errors.csv",
    "results/bert_errors.csv",
    "results/figures/model_metrics.png",
    "results/figures/throughput_comparison.png"
]

for filename in required_files:
    assert Path(filename).exists(), f"Missing: {filename}"

print("Integration smoke test passed")

Integration smoke test passed


In [11]:
print("Classification Results")
print(model_comparison.to_string(index=False))

print("\nPerformance Results")
print(performance_comparison.to_string(index=False))

Classification Results
     model  accuracy  precision   recall       f1
DistilBERT  0.910550   0.897826 0.930180 0.913717
      BERT  0.924312   0.918142 0.934685 0.926339

Performance Results
     model  batch_size  median_time_s  throughput  ms_per_sample
DistilBERT           1      63.050524   13.830178      72.305647
DistilBERT           4      49.674201   17.554384      56.965827
DistilBERT           8      45.763377   19.054538      52.480937
DistilBERT          16      46.369342   18.805529      53.175850
      BERT           1     116.811221    7.465036     133.957822
      BERT           4      83.777509   10.408521      96.075125
      BERT           8      86.310040   10.103112      98.979403
      BERT          16      92.926175    9.383793     106.566715
